# 저장된 결과 분석 (GPU 불필요)

원본 정밀도의 270개 저장 metric을 재집계합니다. 새 학습·추론·가중치 다운로드가 없습니다. PDF 반올림 값은 비교 기준입니다. 검토된 코드 revision을 사용하며 원본 결과를 덮어쓰지 않습니다.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/Han-Youseung/tsfm-zero-few-shot-crossover.git"
SOURCE_REF = "main"  # Replaced with an immutable tested code commit before publication.
repo = Path("tsfm-zero-few-shot-crossover")
if not repo.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(repo)], check=True)
    subprocess.run(
        [
            "git",
            "checkout",
            "--detach",
            (SOURCE_REF if len(SOURCE_REF) == 40 else "origin/" + SOURCE_REF),
        ],
        cwd=repo,
        check=True,
    )
else:
    expected = subprocess.check_output(
        ["git", "rev-parse", (SOURCE_REF if len(SOURCE_REF) == 40 else "origin/" + SOURCE_REF)],
        cwd=repo,
        text=True,
    ).strip()
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain", "--untracked-files=no"], cwd=repo, text=True
    ).strip()
    if actual != expected or dirty:
        raise RuntimeError("Existing checkout differs; preserve it and use a fresh runtime")
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip())

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements/analysis.txt"],
    cwd=repo,
    check=True,
)

In [ ]:
import uuid

output = Path("outputs") / ("colab-analysis-" + uuid.uuid4().hex[:8])
subprocess.run(
    [sys.executable, "-m", "tsfm_crossover.analysis.public_results", "--output", str(output)],
    cwd=repo,
    check=True,
)

In [ ]:
import json

from IPython.display import Image, Markdown, display

display(Markdown((repo / output / "tables.md").read_text(encoding="utf-8")))
display(Image(filename=str(repo / output / "figure1.png")))
verification = json.loads((repo / output / "verification.json").read_text(encoding="utf-8"))
print(json.dumps(verification, ensure_ascii=False, indent=2))

## 선택 분석과 다운로드

mean paired ratios와 improvement from mean errors는 별개입니다. 음영은 seed 최소~최대이며 신뢰구간이 아닙니다. 누락된 seed/비율은 판정 불가로 표시합니다. 0.5%는 전체 학습 기간에서 선택한 윈도 후보 비율이고 관측 기간 0.5%를 뜻하지 않습니다. ETT 네 변형은 두 출처를 공유합니다. 이번 예산·모델·데이터 범위를 넘어 일반화하지 마세요.

In [ ]:
import sys

import matplotlib.pyplot as plt

sys.path.insert(0, str((repo / "src").resolve()))
from tsfm_crossover.analysis.public_results import analyze, load_conditions, plot_curves

MODEL, DATASET = "ttm", "ETTh1"
result = analyze(load_conditions(repo / "results/public/conditions.csv"))
figure = plot_curves(result["curves"], model=MODEL, dataset=DATASET)
display(figure)

plt.close(figure)

In [ ]:
import shutil

archive = shutil.make_archive(str(repo / output), "zip", root_dir=repo / output)
print(archive)
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print("Local Jupyter: use the ZIP path above")